# OOD Detection on Synthetic Perturbations

In the following we demonstrate how to reproduce the OOD detection experiments on the synthetic perturbations from CIFAR-10-C.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score

from sitn.aggregators import MaxQuantile, KDE
from sitn.datasets.cifar10c import CORRUPTIONS
from sitn.utils import construct_results_path, submit_eval_jobs, submit_train_jobs

## Train Flow Matching Model

Note: If you have previously reproduced the cross-dataset experiments, this part can be skipped as the CIFAR-10 model was already trained.

In [ ]:
# Training config
train_cfg = {"dataset_name": "cifar10"}

In [ ]:
# Submit slurm job for training (or use CLI instead)
submit_train_jobs([train_cfg])

Alternatively, via CLI:

`uv run sitn-train --dataset_name cifar10`

## Evaluate Likelihoods

In [ ]:
# ID evaluation configs
eval_cfg_train = {"config": train_cfg, "split_pick": "train"}
eval_cfg_val = {"config": train_cfg, "split_pick": "val"}
eval_cfg_test = {"config": train_cfg, "split_pick": "test"}

In [ ]:
# OOD evaluation configs for the different corruptions
eval_cfgs_ood = [
    {
        "config": train_cfg,
        "eval_dataset_name": "cifar10c",
        "corruptions": [corruption],
        "split_pick": "test",
    }
    for corruption in CORRUPTIONS
]

In [ ]:
# Submit slurm jobs for evaluations (or use CLI instead)
# These jobs should only be submitted after training has completed.
# Note: at the end of training, evaluations are automatically run
# on the train, val, and test splits of the training dataset, so we
# only need to submit evaluation jobs for the OOD datasets.
submit_eval_jobs(eval_cfgs_ood)

Alternatively, via CLI:

`uv run sitn-eval /path/to/training_cfg --eval_dataset_name cifar10c --split_pick test --corruptions brightness`

`uv run sitn-eval /path/to/training_cfg --eval_dataset_name cifar10c --split_pick test --corruptions contrast`

etc.


Note that the training config is created and saved in the output folder when a model is trained.

## Fit OOD Methods

In [ ]:
# Load train and val ID predictions
id_train_preds = pd.read_csv(construct_results_path(**eval_cfg_train, result_type="predictions"))
id_val_preds = pd.read_csv(construct_results_path(**eval_cfg_val, result_type="predictions"))

# Compute entropy estimate for typicality
entropy_estimate = np.mean(id_train_preds["log_likelihood"])

# Fit DoSE
dose = KDE(features=["log_likelihood", "source_log_likelihood", "log_determinant"])
dose.fit(id_train_preds, subsample=10000)

# Fit SITN
sitn = MaxQuantile({"anderson_darling_statistic": True, "ps_cv": True})
sitn.fit(id_val_preds)

## Evaluate OOD Detection Performance

In [ ]:
# Metric configurations
metrics = {
    "log_likelihood": {"label": "Log Likelihood", "higher_is_ood": False},
    "typicality": {"label": "Typicality", "higher_is_ood": True},
    "dose": {"label": "DoSE", "higher_is_ood": False},
    "sitn": {"label": "SITN", "higher_is_ood": True},
}

# Load ID test predictions
id_preds = pd.read_csv(construct_results_path(**eval_cfg_test, result_type="predictions"))
id_preds["train_dataset"] = eval_cfg_test["config"]["dataset_name"]
id_preds["eval_dataset"] = eval_cfg_test["config"]["dataset_name"]

results = []
for eval_cfg_ood in eval_cfgs_ood:
    # Load OOD test predictions
    ood_preds = pd.read_csv(construct_results_path(**eval_cfg_ood, result_type="predictions"))
    ood_preds["train_dataset"] = eval_cfg_ood["config"]["dataset_name"]
    ood_preds["eval_dataset"] = eval_cfg_ood["eval_dataset_name"]

    # Combine ID and OOD predictions
    preds = pd.concat([id_preds.copy(), ood_preds], ignore_index=True)

    # Add baseline and SITN scores
    preds["typicality"] = (preds["log_likelihood"] - entropy_estimate).abs()
    preds["dose"] = dose.score(preds)
    preds["sitn"] = sitn.score(preds)

    # Compute AUROC for each method
    y_true = (preds["eval_dataset"] != preds["train_dataset"]).astype(int)
    for col, meta in metrics.items():
        scores = preds[col].copy()
        if not meta["higher_is_ood"]:
            scores = -scores

        auroc = roc_auc_score(y_true, scores)
        results.append(
            {
                "corruption": eval_cfg_ood["corruptions"][0],
                "metric": meta["label"],
                "AUROC": auroc,
            }
        )

results = pd.DataFrame(results).pivot(index="corruption", columns="metric", values="AUROC")
results = results.reindex(columns=[meta["label"] for meta in metrics.values()])
results


metric,Log Likelihood,Typicality,DoSE,SITN
corruption,,,,
brightness,0.675952,0.616501,0.765300,0.820263
contrast,0.188176,0.687509,0.701502,0.599640
defocus_blur,0.261633,0.578776,0.715301,0.731072
elastic_transform,0.400698,0.464444,0.534624,0.737486
fog,0.331195,0.478544,0.493866,0.635524
frost,0.806059,0.707443,0.910052,0.884353
gaussian_blur,0.196944,0.684453,0.820202,0.751795
gaussian_noise,0.999999,0.996984,1.000000,0.907656
glass_blur,0.841413,0.788287,0.908696,0.953968
